<div style="background:#081420;color:#e6edf3;padding:24px 28px;border-radius:12px;font-family:ui-sans-serif,system-ui,-apple-system,'Segoe UI',Inter,Helvetica,Arial,sans-serif">
  <div style="font-size:12px;font-weight:700;letter-spacing:0.16em;text-transform:uppercase;color:#00d4ff;margin-bottom:6px">Aintropy Solutions</div>
  <div style="font-size:24px;font-weight:600;line-height:1.2">Collision-aware curation, end to end</div>
  <div style="font-size:13.5px;color:#7c8ea1;margin-top:10px;line-height:1.5">Fit an unobstructed-movement model from real telemetry, flag ticks where velocity falls short toward a nearby wall, and upsample them for training.</div>
</div>

## Setup

In [ ]:
%pip install --quiet 'kuration-sdk[gameplay]' matplotlib

## The idea in one paragraph

Good players don't run into walls, so wall-contact events are systematically underrepresented in real gameplay. A world model trained on that footage learns walls are optional. The collision operator uses telemetry (velocity, button state, health, weapon) and map geometry to flag every tick where a player's actual velocity falls short of an unobstructed-movement model in the direction of a nearby wall. Those ticks then get upsampled during training.

The check runs on telemetry and map geometry, not pixels. So collisions in blind spots (a wall behind the player, a crate off-screen) are caught the same way head-on collisions are.

## Fit an unobstructed-movement model from synthetic telemetry

In production, the ramp model is fit from thousands of real gameplay ticks. Here we synthesize a small tick stream with a player pressing forward, then hitting a wall. The unobstructed model learns what forward-key-plus-no-obstacle produces on its own.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
N_TICKS = 400
TICKS_PER_SECOND = 64

# Player holds W for the whole trace. In an obstructed region (ticks 200 to 260),
# a wall in front limits velocity.
btn_forward = np.ones(N_TICKS, dtype=np.uint8)

# Baseline unobstructed velocity: ramps up over 8 ticks then plateaus around 250 u/s
# with small measurement noise. Physics constants approximate CS2 numbers.
target_speed = 250.0
ramp_ticks = 8
ramp_curve = np.clip(np.arange(N_TICKS) / ramp_ticks, 0, 1) * target_speed
noise = np.random.normal(0, 4, size=N_TICKS)
velocity = ramp_curve + noise

# Introduce a wall between ticks 200 and 260: velocity falls to ~40 u/s.
wall_mask = (np.arange(N_TICKS) >= 200) & (np.arange(N_TICKS) < 260)
velocity[wall_mask] = 40 + np.random.normal(0, 3, size=wall_mask.sum())

# Constant health, weapon (irrelevant here, real analyzer conditions on them)
hp = np.full(N_TICKS, 100.0)
weapon = np.zeros(N_TICKS, dtype=np.int32)

# Distance to the wall in the forward direction. Zero outside the wall region.
dist_to_wall = np.full(N_TICKS, np.inf)
dist_to_wall[wall_mask] = np.linspace(50, 5, wall_mask.sum())

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(velocity, color="#00d4ff", linewidth=1.4)
ax.axvspan(200, 260, color="#f87171", alpha=0.15, label="wall in the way")
ax.set_xlabel("tick")
ax.set_ylabel("velocity (u/s)")
ax.set_title("Synthetic velocity trace: W held throughout, wall from tick 200 to 260")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Compute the residual and flag collisions

The unobstructed-movement model says: after 8 ticks of holding W, velocity should be near `target_speed`. The residual is `predicted - measured`. When the residual is large and there's a wall in front, we flag a collision.

In [ ]:
predicted_velocity = np.minimum(
    np.arange(N_TICKS) / ramp_ticks, 1.0
) * target_speed

residual = predicted_velocity - velocity

# Collision heuristic: residual above 100 u/s AND a wall within 50 units in front.
COLLISION_THRESHOLD = 100.0
WALL_PROXIMITY = 50.0
collision = (residual > COLLISION_THRESHOLD) & (dist_to_wall < WALL_PROXIMITY)

n_flagged = int(collision.sum())
print(f"flagged {n_flagged} ticks as collisions ({100 * n_flagged / N_TICKS:.1f}% of ticks)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax1.plot(velocity, color="#00d4ff", linewidth=1.4, label="measured")
ax1.plot(predicted_velocity, color="#4ade80", linewidth=1.2, linestyle="--", label="unobstructed model")
ax1.set_ylabel("velocity (u/s)")
ax1.legend(loc="lower right")
ax1.set_title("Measured vs unobstructed. Gap = wall in the way.")

ax2.plot(residual, color="#7c8ea1", linewidth=1.2, label="residual")
ax2.fill_between(np.arange(N_TICKS), 0, residual, where=collision, color="#f87171", alpha=.7, label="flagged")
ax2.axhline(COLLISION_THRESHOLD, color="#7c8ea1", linestyle=":", linewidth=1)
ax2.set_xlabel("tick")
ax2.set_ylabel("residual (u/s)")
ax2.legend(loc="upper left")
plt.tight_layout()
plt.show()

## From ticks to records

The real analyzer returns one row per tick with `collision`, `residual`, `dist_to_wall`, and `direction`. Downstream `.sample()`, `.bin()`, or `.filter()` compose on top of those records the same way they compose on top of any other measured feature.

In [ ]:
records = [
    {
        "tick": int(t),
        "collision": bool(collision[t]),
        "residual": float(residual[t]),
        "dist_to_wall": float(dist_to_wall[t]) if np.isfinite(dist_to_wall[t]) else None,
        "direction": "front",
    }
    for t in range(N_TICKS)
]
print("first flagged tick record:", next(r for r in records if r["collision"]))

## Extending to real footage

The full pipeline against the DIAMOND CS:GO dataset lives in the SDK. You supply the `.npy` telemetry arrays and a `.bsp` map file, register the operator, and let it walk every tick:

```python
import kuration
from kuration_collision import fit_ramp_models, load_directional_arrays, save_ramp_models, register

register()
sources = [load_directional_arrays(p) for p in npy_paths]
models = fit_ramp_models([(s["btn_mask"], s["velocity"], s["hp"], s["weapon"]) for s in sources])
save_ramp_models(models, "ramp_models.json")

analyzer = kuration.get_analyzer(
    "aintropy/collision-directional",
    map_for_source={p: "de_dust2" for p in npy_paths},
    map_registry={"de_dust2": ["/path/to/de_dust2.bsp"]},
    ramp_models_path="ramp_models.json",
)

for path in npy_paths:
    analyzer.process_source(path, **load_directional_arrays(path))

records = analyzer.to_records()
```

The public `de_dust2` map file is on the Steam Community. The DIAMOND CS:GO telemetry is in the [`TeaPearce/CounterStrike_Deathmatch`](https://huggingface.co/datasets/TeaPearce/CounterStrike_Deathmatch) dataset.